# 📡 Multipath Propagation & Friis Transfer Equation
### Interactive Teaching Notebook — DHBW Mobile Communications
---
This notebook visualises the **multipath propagation** concepts from the lecture slide:
- ✅ **Direct path** (LOS – Line of Sight, green)
- 🔵 **Diffraction** over a house (Beugung)
- 🔴 **Scattering** by a tree (Streuung)
- ⚫ **Reflection** from a building

Use the sliders below to explore how transmit power, frequency, distance and obstacle losses
affect the **Friis received power** and the multipath-impaired total received power.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.patches import FancyArrowPatch, Arc, FancyBboxPatch, Polygon
from matplotlib.lines import Line2D
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries loaded successfully.")


✅ All libraries loaded successfully.


In [2]:
# ─────────────────────────────────────────────────────────────
#  SCENE DRAWING  — positions are in "scene units" (0..10, 0..6)
# ─────────────────────────────────────────────────────────────

def draw_tower(ax, x, y, color='#444'):
    """Draw a simple base-station tower."""
    # mast
    ax.plot([x, x], [y, y+1.5], color=color, lw=3, zorder=5)
    # cross-arms
    for h in [y+0.4, y+0.8, y+1.2]:
        ax.plot([x-0.25, x+0.25], [h, h], color=color, lw=2, zorder=5)
    # antenna arcs
    for r in [0.15, 0.30, 0.45]:
        arc = Arc((x, y+1.6), r, r*0.6, angle=0, theta1=10, theta2=170,
                  color='steelblue', lw=1.5, zorder=5)
        ax.add_patch(arc)
    ax.text(x, y-0.25, 'BS (Tx)', ha='center', va='top', fontsize=8,
            fontweight='bold', color=color)


def draw_house(ax, x, y, w=1.0, h=0.7, color='#c0392b'):
    """Simple house: rectangle + triangle roof."""
    body = FancyBboxPatch((x-w/2, y), w, h,
                          boxstyle='square,pad=0',
                          fc='#f5cba7', ec='#884400', lw=1.5, zorder=4)
    ax.add_patch(body)
    roof = Polygon([[x-w/2, y+h], [x+w/2, y+h], [x, y+h+h*0.6]],
                   closed=True, fc=color, ec='#7b241c', lw=1.5, zorder=4)
    ax.add_patch(roof)
    # chimney
    ax.add_patch(FancyBboxPatch((x+0.1, y+h+h*0.3), 0.12, 0.22,
                                boxstyle='square,pad=0',
                                fc='#95a5a6', ec='#666', lw=1, zorder=5))
    ax.text(x, y-0.22, 'House', ha='center', fontsize=7.5, color='#884400')


def draw_building(ax, x, y, w=0.9, h=1.8, color='#d4a96a'):
    """Tall building with windows."""
    body = FancyBboxPatch((x-w/2, y), w, h,
                          boxstyle='square,pad=0',
                          fc=color, ec='#8B6914', lw=2, zorder=4)
    ax.add_patch(body)
    # windows grid
    for row in range(3):
        for col in range(2):
            wx = x - w/2 + 0.15 + col*0.32
            wy = y + 0.2 + row*0.48
            ax.add_patch(FancyBboxPatch((wx, wy), 0.18, 0.22,
                                        boxstyle='square,pad=0',
                                        fc='lightcyan', ec='#555', lw=0.8, zorder=5))
    ax.text(x, y-0.22, 'Building', ha='center', fontsize=7.5, color='#8B6914')


def draw_tree(ax, x, y, color='#27ae60'):
    """Simple tree: trunk + foliage circle."""
    ax.plot([x, x], [y, y+0.5], color='#795548', lw=3, zorder=4)
    foliage = plt.Circle((x, y+0.9), 0.45, fc=color, ec='#1a7a40', lw=1.5, zorder=4)
    ax.add_patch(foliage)
    ax.text(x, y-0.22, 'Tree', ha='center', fontsize=7.5, color=color)


def draw_mobile(ax, x, y):
    """Simple tram / mobile receiver rectangle."""
    body = FancyBboxPatch((x-0.65, y), 1.3, 0.5,
                          boxstyle='round,pad=0.05',
                          fc='#d5e8d4', ec='#2d6a4f', lw=2, zorder=4)
    ax.add_patch(body)
    # windows
    for wx in [x-0.35, x+0.05]:
        ax.add_patch(FancyBboxPatch((wx, y+0.1), 0.28, 0.25,
                                    boxstyle='square,pad=0',
                                    fc='lightblue', ec='#2d6a4f', lw=0.8, zorder=5))
    # wheels
    for wx in [x-0.4, x+0.3]:
        ax.add_patch(plt.Circle((wx, y), 0.1, fc='#555', ec='#333', lw=1, zorder=5))
    ax.text(x, y-0.35, 'MS (Rx)', ha='center', fontsize=8,
            fontweight='bold', color='#2d6a4f')


def curved_arrow(ax, x0, y0, x1, y1, color, lw=2,
                 bend=0.3, label='', label_offset=(0,0)):
    """Draw a curved arrow using FancyArrowPatch with arc3 connection style."""
    arrow = FancyArrowPatch(
        (x0, y0), (x1, y1),
        connectionstyle=f'arc3,rad={bend}',
        arrowstyle='->', color=color, lw=lw,
        mutation_scale=14, zorder=6,
        path_effects=[pe.Stroke(linewidth=lw+1, foreground='white'), pe.Normal()]
    )
    ax.add_patch(arrow)
    if label:
        mx = (x0+x1)/2 + label_offset[0]
        my = (y0+y1)/2 + label_offset[1]
        ax.text(mx, my, label, color=color, fontsize=8.5,
                fontweight='bold', ha='center', va='center',
                bbox=dict(fc='white', ec=color, alpha=0.7, pad=1.5, boxstyle='round'))


print("✅ Scene drawing helpers defined.")


✅ Scene drawing helpers defined.


In [3]:
# ─────────────────────────────────────────────────────────────
#  FRIIS TRANSFER EQUATION  (free-space, LOS)
# ─────────────────────────────────────────────────────────────

c0 = 3e8  # speed of light [m/s]

def friis_power_dBm(P_tx_dBm, G_tx_dBi, G_rx_dBi, freq_MHz, dist_m):
    """
    P_Rx = P_Tx * G_Tx * (c0 / (4*pi*r*f))^2 * G_Rx
    Returns received power in dBm.
    """
    f = freq_MHz * 1e6
    lambda_ = c0 / f
    FSPL_dB = 20*np.log10(4*np.pi*dist_m/lambda_)  # free-space path loss [dB]
    P_rx_dBm = P_tx_dBm + G_tx_dBi - FSPL_dB + G_rx_dBi
    return P_rx_dBm, FSPL_dB

def multipath_received_dBm(P_los_dBm, losses_dB):
    """
    Combine LOS with multipath components (power addition in linear domain).
    losses_dB: list of extra losses [dB] for each additional path.
    """
    P_total_lin = 10**(P_los_dBm/10)
    for loss in losses_dB:
        P_total_lin += 10**((P_los_dBm - loss)/10)
    return 10*np.log10(P_total_lin)


print("✅ Friis equation functions defined.")


✅ Friis equation functions defined.


In [4]:
# ─────────────────────────────────────────────────────────────
#  INTERACTIVE WIDGET
# ─────────────────────────────────────────────────────────────

style  = {'description_width': '155px'}
layout = widgets.Layout(width='480px')

# ── Sliders ──
sl_ptx   = widgets.FloatSlider(value=30, min=0,    max=50,   step=1,
    description='P_tx [dBm]:', style=style, layout=layout)
sl_freq  = widgets.FloatSlider(value=1800, min=700, max=6000, step=100,
    description='Frequency [MHz]:', style=style, layout=layout)
sl_dist  = widgets.FloatSlider(value=500, min=10,  max=5000, step=10,
    description='Distance r [m]:', style=style, layout=layout)
sl_gtx   = widgets.FloatSlider(value=15, min=0,    max=30,   step=1,
    description='G_Tx [dBi]:', style=style, layout=layout)
sl_grx   = widgets.FloatSlider(value=0,  min=0,    max=15,   step=1,
    description='G_Rx [dBi]:', style=style, layout=layout)
sl_diff  = widgets.FloatSlider(value=10, min=0,    max=30,   step=1,
    description='Diffraction loss [dB]:', style=style, layout=layout)
sl_scat  = widgets.FloatSlider(value=15, min=0,    max=30,   step=1,
    description='Scattering loss [dB]:', style=style, layout=layout)
sl_refl  = widgets.FloatSlider(value=8,  min=0,    max=30,   step=1,
    description='Reflection loss [dB]:', style=style, layout=layout)

sliders = [sl_ptx, sl_freq, sl_dist, sl_gtx, sl_grx,
           sl_diff, sl_scat, sl_refl]

out = widgets.Output()

def update(_=None):
    with out:
        clear_output(wait=True)

        P_tx   = sl_ptx.value
        freq   = sl_freq.value
        dist   = sl_dist.value
        G_tx   = sl_gtx.value
        G_rx   = sl_grx.value
        L_diff = sl_diff.value
        L_scat = sl_scat.value
        L_refl = sl_refl.value

        P_los, FSPL = friis_power_dBm(P_tx, G_tx, G_rx, freq, dist)
        P_total = multipath_received_dBm(P_los, [L_diff, L_scat, L_refl])
        lambda_m = c0 / (freq*1e6)

        # ── Figure layout ────────────────────────────────────────
        fig = plt.figure(figsize=(15, 10))
        gs  = fig.add_gridspec(2, 2, hspace=0.45, wspace=0.35,
                               left=0.05, right=0.97, top=0.92, bottom=0.08)
        ax_scene  = fig.add_subplot(gs[0, :])   # full-width top
        ax_pvsr   = fig.add_subplot(gs[1, 0])   # power vs distance
        ax_bar    = fig.add_subplot(gs[1, 1])   # bar comparison

        # ════════════════════════════════════════════════════════
        # PANEL 1 – SCENE
        # ════════════════════════════════════════════════════════
        ax = ax_scene
        ax.set_xlim(0, 10)
        ax.set_ylim(-0.5, 6.2)
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_facecolor('#f0f4f8')
        fig.patch.set_facecolor('#fafafa')

        # Sky gradient rectangle
        sky = FancyBboxPatch((0, 0), 10, 6.2, boxstyle='square,pad=0',
                              fc='#dbe9f7', ec='none', zorder=0)
        ax.add_patch(sky)
        # Ground
        ground = FancyBboxPatch((0, -0.5), 10, 0.6, boxstyle='square,pad=0',
                                 fc='#a8d5a2', ec='none', zorder=1)
        ax.add_patch(ground)
        ax.axhline(0.08, color='#6aab6a', lw=1.5, zorder=2)

        # ── Obstacles ──
        TX_pos  = (0.8, 0.1)   # tower base
        TX_top  = (0.8, 1.7)   # transmit point (antenna tip)
        RX_pos  = (7.2, 0.28)  # tram centre-top

        draw_tower(ax, TX_pos[0], TX_pos[1])
        draw_house(ax, 3.5, 0.1, w=1.1, h=0.8)
        draw_tree(ax,  2.0, 0.1)
        draw_building(ax, 8.8, 0.1, w=1.1, h=2.0)
        draw_mobile(ax, RX_pos[0], RX_pos[1]-0.28)

        # ── Propagation paths ──
        # 1. Direct / LOS path (green solid) — only if roughly LOS
        curved_arrow(ax, TX_top[0], TX_top[1],
                     RX_pos[0], RX_pos[1]+0.22,
                     color='#27ae60', lw=2.5, bend=-0.05,
                     label='', label_offset=(0, 0.55))

        # 2. Diffraction over house (blue, arcs over roof)
        house_tip = (3.5, 0.1+0.8+0.8*0.6)   # top of roof  ≈ (3.5, 1.38)
        curved_arrow(ax, TX_top[0], TX_top[1],
                     house_tip[0], house_tip[1]+0.1,
                     color='#2980b9', lw=2, bend=-0.15)
        curved_arrow(ax, house_tip[0], house_tip[1]+0.1,
                     RX_pos[0], RX_pos[1]+0.22,
                     color='#2980b9', lw=2, bend=-0.15,
                     label='', label_offset=(0.3, 0.5))

        # 3. Scattering from tree (red, multi-bounce arrows)
        tree_top = (2.0, 0.1+0.5+0.9)         # foliage centre ≈ (2.0, 1.5)
        curved_arrow(ax, TX_top[0], TX_top[1],
                     tree_top[0], tree_top[1],
                     color='#c0392b', lw=2, bend=0.2)
        # scattered rays fan out toward receiver
        for bend_val, loff in [(-0.2, (-0.4, 0.4)), (0.1, (0.1, -0.3))]:
            curved_arrow(ax, tree_top[0], tree_top[1],
                         RX_pos[0], RX_pos[1]+0.22,
                         color='#c0392b', lw=1.5, bend=bend_val)
        ax.text(tree_top[0]-0.1, tree_top[1]+0.3,
                '', color='#c0392b', fontsize=8,
                fontweight='bold', ha='center',
                bbox=dict(fc='white', ec='#c0392b', alpha=0.7, pad=1.5, boxstyle='round'))

        # 4. Reflection from building (black, bounce off wall)
        bldg_wall = (8.8 - 1.1/2 - 0.05, 1.2)   # left face of building
        curved_arrow(ax, TX_top[0], TX_top[1],
                     bldg_wall[0], bldg_wall[1],
                     color='#2c3e50', lw=2, bend=-0.25)
        curved_arrow(ax, bldg_wall[0], bldg_wall[1],
                     RX_pos[0]+0.1, RX_pos[1]+0.22,
                     color='#2c3e50', lw=2, bend=0.35,
                     label='', label_offset=(-0.9, 0.6))

        # ── Power annotation box ──
        info = (f"P_Tx = {P_tx:.0f} dBm\n"
                f"f = {freq:.0f} MHz\n"
                f"λ = {lambda_m*100:.1f} cm\n"
                f"r = {dist:.0f} m\n"
                f"FSPL = {FSPL:.1f} dB\n"
                f"P_Rx(LOS) = {P_los:.1f} dBm\n"
                f"P_Rx(total) = {P_total:.1f} dBm")
        ax.text(5, 5.85, info, ha='center', va='top', fontsize=9,
                bbox=dict(fc='#ffffcc', ec='#cccc00', alpha=0.9, pad=4,
                          boxstyle='round,pad=0.4'))

        # ── Legend ──
        legend_elements = [
            Line2D([0],[0], color='#27ae60', lw=2.5, label='Direct path (LOS)'),
            Line2D([0],[0], color='#2980b9', lw=2,   label='Diffraction (Beugung)'),
            Line2D([0],[0], color='#c0392b', lw=2,   label='Scattering (Streuung)'),
            Line2D([0],[0], color='#2c3e50', lw=2,   label='Reflection'),
        ]
        ax.legend(handles=legend_elements, loc='upper right',
                  fontsize=8.5, framealpha=0.85)
        ax.set_title('Multipath Propagation for Radio Signals — Interactive Scene',
                     fontsize=13, fontweight='bold', pad=8)

        # ════════════════════════════════════════════════════════
        # PANEL 2 – P_Rx vs Distance
        # ════════════════════════════════════════════════════════
        ax2 = ax_pvsr
        r_arr = np.logspace(1, np.log10(5000), 300)
        P_los_arr   = np.array([friis_power_dBm(P_tx, G_tx, G_rx, freq, r)[0]
                                for r in r_arr])
        #P_total_arr = np.array([multipath_received_dBm(p,
        #                            [L_diff, L_scat, L_refl])
        #                        for p in P_los_arr])

        ax2.semilogx(r_arr, P_los_arr,   color='#27ae60', lw=2,
                     label='P_Rx LOS (Friis)')
        #ax2.semilogx(r_arr, P_total_arr, color='#8e44ad', lw=2,
        #             linestyle='--', label='P_Rx with multipath')
        ax2.axvline(dist, color='gray', lw=1.2, linestyle=':', alpha=0.8)
        ax2.axhline(P_los, color='#27ae60', lw=1, linestyle=':', alpha=0.6)
        ax2.scatter([dist], [P_los],   color='#27ae60', s=60, zorder=5)
        ax2.scatter([dist], [P_total], color='#8e44ad', s=60, zorder=5)
        # slope annotation
        ax2.text(0.97, 0.06,
                 r'$P_{Rx} \propto \frac{1}{r^2}$',
                 transform=ax2.transAxes, ha='right', fontsize=11,
                 color='#27ae60',
                 bbox=dict(fc='white', ec='#27ae60', alpha=0.8, pad=3,
                           boxstyle='round'))
        ax2.set_xlabel('Distance r [m]', fontsize=10)
        ax2.set_ylabel('Received Power [dBm]', fontsize=10)
        ax2.set_title('P_Rx vs Distance (log scale)', fontsize=11,
                      fontweight='bold')
        ax2.legend(fontsize=8.5)
        ax2.grid(True, which='both', linestyle='--', alpha=0.4)
        ax2.set_facecolor('#f9f9f9')

        # ════════════════════════════════════════════════════════
        # PANEL 3 – Bar chart: path contributions
        # ════════════════════════════════════════════════════════
        ax3 = ax_bar
        path_names  = ['LOS(direct)', 'Diffraction(Beugung)','Scattering(Streuung)', 'Reflection']
        path_losses = [0, L_diff, L_scat, L_refl]
        path_powers = [P_los - loss for loss in path_losses]
        bar_colors  = ['#27ae60', '#2980b9', '#c0392b', '#2c3e50']

        bars = ax3.bar(path_names, path_powers, color=bar_colors,
                       edgecolor='white', linewidth=1.5, zorder=3)
        for bar, val in zip(bars, path_powers):
            ax3.text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + 0.5,
                     f'{val:.1f}\ndBm', ha='center', va='bottom',
                     fontsize=8, fontweight='bold')
        ax3.set_ylabel('Path Received Power [dBm]', fontsize=10)
        ax3.set_title('Individual Path Power Contributions', fontsize=11,
                      fontweight='bold')
        ax3.set_ylim(min(path_powers)-8, max(path_powers)+8)
        ax3.axhline(-100, color='red', lw=1.2, linestyle='--',
                    label='Typical noise floor −100 dBm', alpha=0.7)
        ax3.legend(fontsize=8, loc='lower right')
        ax3.grid(True, axis='y', linestyle='--', alpha=0.4)
        ax3.set_facecolor('#f9f9f9')

        plt.suptitle(
            r'Friis:  $P_{Rx} = P_{Tx} \cdot G_{Tx,LOS} \cdot'
            r'\left(\dfrac{c_0}{4\pi r f}\right)^2 \cdot G_{Rx,LOS}$',
            fontsize=13, y=0.99)

        plt.show()

for sl in sliders:
    sl.observe(update, names='value')

ui = widgets.VBox([
    widgets.HTML("<h3>⚙️ System Parameters</h3>"),
    widgets.HBox([
        widgets.VBox([sl_ptx, sl_freq, sl_dist, sl_gtx, sl_grx],
                     layout=widgets.Layout(margin='0 30px 0 0')),
        widgets.VBox([
            widgets.HTML("<b>🏠 Obstacle Extra Losses</b>"),
            sl_diff, sl_scat, sl_refl
        ])
    ]),
    out
])

display(ui)
update()


In [5]:
# ─────────────────────────────────────────────────────────────
#  STANDALONE: print Friis equation breakdown
# ─────────────────────────────────────────────────────────────

def explain_friis(P_tx_dBm=30, G_tx_dBi=15, G_rx_dBi=0,
                  freq_MHz=1800, dist_m=500):
    f   = freq_MHz * 1e6
    lam = c0 / f
    FSPL_linear = (4*np.pi*dist_m/lam)**2
    FSPL_dB     = 10*np.log10(FSPL_linear)
    G_tx_lin    = 10**(G_tx_dBi/10)
    G_rx_lin    = 10**(G_rx_dBi/10)
    P_tx_W      = 10**((P_tx_dBm-30)/10)
    P_rx_W      = P_tx_W * G_tx_lin / FSPL_linear * G_rx_lin
    P_rx_dBm    = 10*np.log10(P_rx_W*1000)

    print("=" * 55)
    print("  FRIIS TRANSFER EQUATION — Step-by-Step")
    print("=" * 55)
    print(f"  P_Tx          = {P_tx_dBm} dBm  ({P_tx_W*1000:.2f} mW)")
    print(f"  G_Tx          = {G_tx_dBi} dBi  (×{G_tx_lin:.1f} linear)")
    print(f"  Frequency     = {freq_MHz} MHz  →  λ = {lam*100:.2f} cm")
    print(f"  Distance r    = {dist_m} m")
    print(f"  FSPL          = {FSPL_dB:.2f} dB  (×{FSPL_linear:.2e} linear)")
    print(f"  G_Rx          = {G_rx_dBi} dBi  (×{G_rx_lin:.1f} linear)")
    print("-" * 55)
    print(f"  P_Rx (LOS)    = {P_rx_dBm:.2f} dBm  ({P_rx_W*1e9:.3f} nW)")
    print("=" * 55)
    print(f"  Note: doubling r → P_Rx drops by ~6 dB  (1/r² law)")
    print(f"  Doubling f       → P_Rx drops by ~6 dB  (shorter λ)")

explain_friis()


  FRIIS TRANSFER EQUATION — Step-by-Step
  P_Tx          = 30 dBm  (1000.00 mW)
  G_Tx          = 15 dBi  (×31.6 linear)
  Frequency     = 1800 MHz  →  λ = 16.67 cm
  Distance r    = 500 m
  FSPL          = 91.53 dB  (×1.42e+09 linear)
  G_Rx          = 0 dBi  (×1.0 linear)
-------------------------------------------------------
  P_Rx (LOS)    = -46.53 dBm  (22.250 nW)
  Note: doubling r → P_Rx drops by ~6 dB  (1/r² law)
  Doubling f       → P_Rx drops by ~6 dB  (shorter λ)
